In [42]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [43]:
%cd /content/drive/MyDrive/Colab Notebooks/capstone

/content/drive/MyDrive/Colab Notebooks/capstone


In [44]:
import os
import time
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

In [45]:
data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/capstone/Skincare Treatment Dataset.csv')
print('Ukuran data:', data.shape)
print()
print("Informasi Kolom:")
print(data.info())
print()
print("Pratinjau Data:")
display(data.head())

Ukuran data: (1120, 9)

Informasi Kolom:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1120 entries, 0 to 1119
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Age_Group       1120 non-null   object
 1   Skin_Type       1120 non-null   object
 2   Skin_Subtype    1120 non-null   object
 3   Sensitivity     1120 non-null   object
 4   Concern         1120 non-null   object
 5   Internal_Type   1120 non-null   object
 6   Ingredients     1120 non-null   object
 7   Concentrations  1120 non-null   object
 8   Effects         1120 non-null   object
dtypes: object(9)
memory usage: 78.9+ KB
None

Pratinjau Data:


,Age_Group,Skin_Type,Skin_Subtype,Sensitivity,Concern,Internal_Type,Ingredients,Concentrations,Effects
0,14-18,Normal,Normal to Dry,Yes,Acne,Comedonal,Zinc PCA + Benzoyl Peroxide + Salicylic Acid,0.5% + 2% + 5%,"Controls sebum production, Kills acne-causing ..."
1,14-18,Normal,Normal to Dry,Yes,Acne,Inflammatory,Zinc PCA + Benzoyl Peroxide + Niacinamide,0.5% + 5% + 1%,"Controls sebum production, Kills acne-causing ..."
2,14-18,Normal,Normal to Dry,Yes,Acne,Nodular,Zinc PCA + Azelaic Acid + Salicylic Acid,10% + 1% + 5%,"Controls sebum production, Reduces acne and im..."
3,14-18,Normal,Normal to Dry,Yes,Acne,Cystic,Benzoyl Peroxide + Niacinamide + Azelaic Acid,10% + 1% + 8%,"Kills acne-causing bacteria, Reduces inflammat..."
4,14-18,Normal,Normal to Dry,Yes,Dark Circles,Pigmented,Hyaluronic Acid + Caffeine + Vitamin C,3% + 2% + 0.5%,"Hydrates and plumps the skin, Reduces puffines..."


menentukan fitur dan target lalu membagi data

In [46]:
# Menentukan Fitur dan Target disesuaikan dengan nama kolom kapital di dataset kamu
features = ['Age_Group', 'Skin_Type', 'Sensitivity', 'Concern']
target_ingredients = 'Ingredients'
target_effects = 'Effects'

# One-Hot Encoding untuk fitur kategorikal
X = pd.get_dummies(data[['Age_Group', 'Skin_Type', 'Sensitivity', 'Concern']], drop_first=True)
y = data['Ingredients'] # Target bahan aktif

# Membagi data menjadi train dan test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Train shape:', X_train.shape, '| Test shape:', X_test.shape)
print("Kelas Target unik:", y.unique()[:10], "... (total:", len(y.unique()), "kelas)")

Train shape: (896, 17) | Test shape: (224, 17)
Kelas Target unik: ['Zinc PCA + Benzoyl Peroxide + Salicylic Acid'
 'Zinc PCA + Benzoyl Peroxide + Niacinamide'
 'Zinc PCA + Azelaic Acid + Salicylic Acid'
 'Benzoyl Peroxide + Niacinamide + Azelaic Acid'
 'Hyaluronic Acid + Caffeine + Vitamin C'
 'Vitamin C + Niacinamide + Caffeine'
 'Alpha Arbutin + Niacinamide + Vitamin C'
 'Vitamin C + Lactic Acid + Green Tea Extract'
 'Alpha Arbutin + Vitamin C + Niacinamide'
 'Azelaic Acid + Salicylic Acid + Green Tea Extract'] ... (total: 504 kelas)


standarisasi fitur tanpa data leakage

In [47]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Contoh data train hasil skala (3 baris pertama):")
print(X_train_scaled[:3])

Contoh data train hasil skala (3 baris pertama):
[[-0.48741057 -0.5013947  -0.4996512  -0.51530255 -0.53778121 -0.5962566
  -0.59281775 -1.0044743  -0.41196213 -0.2773501  -0.27265459 -0.28428905
  -0.2773501  -0.27967662 -0.26789986 -0.27967662 -0.28428905]
 [-0.48741057 -0.5013947   2.00139616 -0.51530255 -0.53778121  1.67713027
  -0.59281775  0.99554563 -0.41196213 -0.2773501  -0.27265459 -0.28428905
  -0.2773501  -0.27967662 -0.26789986  3.57555805 -0.28428905]
 [-0.48741057  1.99443671 -0.4996512  -0.51530255 -0.53778121  1.67713027
  -0.59281775  0.99554563 -0.41196213 -0.2773501  -0.27265459 -0.28428905
  -0.2773501   3.57555805 -0.26789986 -0.27967662 -0.28428905]]


membuat dan melatih SVM Linear

In [48]:
mulai = time.time()
svm_linear = SVC(kernel='linear', C=1.0, random_state=42)
svm_linear.fit(X_train_scaled, y_train)
waktu_linear = time.time() - mulai

print('Jumlah support vector (Linear):', svm_linear.n_support_.sum())
print(f'Waktu training             : {waktu_linear:.4f} detik')

Jumlah support vector (Linear): 896
Waktu training             : 0.5041 detik


membuat dan melatih SVM RBF

In [49]:
mulai = time.time()
svm_rbf = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm_rbf.fit(X_train_scaled, y_train)
waktu_rbf = time.time() - mulai

print('Jumlah support vector (RBF):', svm_rbf.n_support_.sum())
print(f'Waktu training             : {waktu_rbf:.4f} detik')

Jumlah support vector (RBF): 896
Waktu training             : 0.5059 detik


melakukan prediksi

In [50]:
y_pred_linear = svm_linear.predict(X_test_scaled)
y_pred_rbf = svm_rbf.predict(X_test_scaled)

hasil_prediksi = pd.DataFrame({
    'Aktual': y_test.values[:10],
    'Prediksi Linear': y_pred_linear[:10],
    'Prediksi RBF': y_pred_rbf[:10]
})
hasil_prediksi

,Aktual,Prediksi Linear,Prediksi RBF
0,Hyaluronic Acid + Licorice Extract + Niacinamide,Licorice Extract + Vitamin C + Vitamin K,Licorice Extract + Vitamin C + Vitamin K
1,Zinc PCA + Benzoyl Peroxide + Salicylic Acid,Green Tea Extract + Niacinamide + Salicylic Acid,Green Tea Extract + Niacinamide + Salicylic Acid
2,Alpha Arbutin + Licorice Extract + Aloe Vera,Alpha Arbutin + Licorice Extract + Vitamin C,Alpha Arbutin + Niacinamide + Licorice Extract
3,Licorice Extract + Aloe Vera + Niacinamide,Alpha Arbutin + Niacinamide + Aloe Vera,Alpha Arbutin + Niacinamide + Licorice Extract
4,Caffeine + Vitamin C + Hyaluronic Acid,Licorice Extract + Vitamin C + Vitamin K,Licorice Extract + Caffeine + Niacinamide
5,Willow Bark Extract + Glycolic Acid + Niacinamide,Zinc PCA + Niacinamide + Glycolic Acid,Zinc PCA + Niacinamide + Glycolic Acid
6,Benzoyl Peroxide + Azelaic Acid + Niacinamide,Azelaic Acid + Benzoyl Peroxide + Niacinamide,Green Tea Extract + Salicylic Acid + Niacinamide
7,Niacinamide + Coenzyme Q10 + Peptides,Coenzyme Q10 + Hyaluronic Acid + Niacinamide,Coenzyme Q10 + Hyaluronic Acid + Niacinamide
8,Green Tea Extract + Azelaic Acid + Salicylic Acid,Green Tea Extract + Niacinamide + Salicylic Acid,Salicylic Acid + Niacinamide + Zinc PCA
9,Niacinamide + Zinc PCA + Azelaic Acid,Salicylic Acid + Zinc PCA + Green Tea Extract,Salicylic Acid + Zinc PCA + Green Tea Extract


menghitung akurasi

In [51]:
# Pastikan nama kelas dikonversi ke string agar classification_report tidak error
nama_kelas = sorted(list(y_test.unique()))
nama_kelas_str = [str(k) for k in nama_kelas]

# Menyamakan panjang data prediksi dan testing
y_pred_l_fixed = y_pred_linear[:len(y_test)]
y_pred_r_fixed = y_pred_rbf[:len(y_test)]

# Menampilkan akurasi
print(f'Akurasi SVM Linear: {accuracy_score(y_test, y_pred_l_fixed):.4f}')
print(f'Akurasi SVM RBF   : {accuracy_score(y_test, y_pred_r_fixed):.4f}')

# Classification Report (menggunakan labels agar jumlah target_names cocok secara presisi)
print('\n=== SVM Linear ===')
print(classification_report(y_test, y_pred_l_fixed, labels=nama_kelas, target_names=nama_kelas_str, digits=4, zero_division=0))

print('=== SVM RBF ===')
print(classification_report(y_test, y_pred_r_fixed, labels=nama_kelas, target_names=nama_kelas_str, digits=4, zero_division=0))

# Confusion Matrix
for nama, pred in [('SVM Linear', y_pred_l_fixed), ('SVM RBF', y_pred_r_fixed)]:
    cm = pd.DataFrame(confusion_matrix(y_test, pred, labels=nama_kelas),
                      index=[f'aktual {k}' for k in nama_kelas],
                      columns=[f'pred {k}' for k in nama_kelas])
    print(f'\nConfusion matrix {nama} (baris = aktual, kolom = prediksi)')
    print(cm.head(10)) # Tampilkan 10 baris pertama saja agar output tidak terlalu panjang

Akurasi SVM Linear: 0.0134
Akurasi SVM RBF   : 0.0089

=== SVM Linear ===
                                                           precision    recall  f1-score   support

        Aloe Vera + Centella Asiatica + Green Tea Extract     0.0000    0.0000    0.0000         1
              Aloe Vera + Chamomile Extract + Niacinamide     0.0000    0.0000    0.0000         2
             Aloe Vera + Licorice Extract + Alpha Arbutin     0.0000    0.0000    0.0000         1
               Aloe Vera + Licorice Extract + Niacinamide     0.0000    0.0000    0.0000         1
                 Aloe Vera + Licorice Extract + Vitamin C     0.0000    0.0000    0.0000         2
               Aloe Vera + Niacinamide + Licorice Extract     0.0000    0.0000    0.0000         1
                  Alpha Arbutin + Aloe Vera + Niacinamide     0.0000    0.0000    0.0000         1
                 Alpha Arbutin + Kojic Acid + Niacinamide     0.0000    0.0000    0.0000         1
             Alpha Arbutin + Licor

membandingkan hasil kedua model dalam tabel

In [52]:
hasil = pd.DataFrame({
    'Model': [
        'SVM Linear',
        'SVM RBF'
    ],
    'Akurasi': [
        accuracy_score(y_test, y_pred_l_fixed),
        accuracy_score(y_test, y_pred_r_fixed)
    ],
    'Precision': [
        precision_score(y_test, y_pred_l_fixed, average='macro', zero_division=0),
        precision_score(y_test, y_pred_r_fixed, average='macro', zero_division=0)
    ],
    'Recall': [
        recall_score(y_test, y_pred_l_fixed, average='macro', zero_division=0),
        recall_score(y_test, y_pred_r_fixed, average='macro', zero_division=0)
    ],
    'F1-score': [
        f1_score(y_test, y_pred_l_fixed, average='macro', zero_division=0),
        f1_score(y_test, y_pred_r_fixed, average='macro', zero_division=0)
    ],
    'Waktu training (detik)': [
        waktu_linear,
        waktu_rbf
    ]
}).round(4)

hasil

,Model,Akurasi,Precision,Recall,F1-score,Waktu training (detik)
0,SVM Linear,0.0134,0.0109,0.0109,0.0109,0.5041
1,SVM RBF,0.0089,0.0054,0.0100,0.0057,0.5059


menyimpan model dan scaler

In [53]:
# Menyimpan model langsung di direktori saat ini
joblib.dump(svm_linear, 'model_svm_linear_skincare.pkl')
joblib.dump(svm_rbf, 'model_svm_rbf_skincare.pkl')
joblib.dump(scaler, 'scaler_svm_skincare.pkl')

print('Model SVM Linear, SVM RBF, dan scaler khusus SKINCARE berhasil disimpan langsung di folder capstone!')

Model SVM Linear, SVM RBF, dan scaler khusus SKINCARE berhasil disimpan langsung di folder capstone!


memanggil model dan menguji dengan input pengguna

In [62]:
# Memuat model & scaler skincare langsung dari direktori aktif
model_linear = joblib.load('model_svm_linear_skincare.pkl')
model_rbf = joblib.load('model_svm_rbf_skincare.pkl')
scaler_baru = joblib.load('scaler_svm_skincare.pkl')

print("=== INPUT KARAKTERISTIK KULIT ANDA ===")
print("Pilihan Age Group: '14-18', '19-24', '25-36', '37-45', '45+'")
age_input = input("Masukkan Age Group: ").strip()

print("Pilihan Sensitivity: 'No', 'Yes'")
sensitivity_input = input("Apakah sensitif? (Yes/No): ").strip().title()

print("Pilihan Concern: 'Acne', 'Aging', 'Blackheads', 'Dark Spots', 'Dull Skin', 'Pores', 'Redness', 'dark circles', 'hyperpigmentation'")
concern_input = input("Masukkan Concern: ").strip().title()

print("Pilihan Effects: silakan isi bebas (contoh: 'Hydrating', 'Soothing', 'Brightening')")
effects_input = input("Masukkan Effects yang dicari: ").strip().title()

# Membuat dataframe dasar sesuai dengan kolom training yang menggunakan huruf kapital
input_data = pd.DataFrame([{
    'Age_Group': age_input,
    'Skin_Type': 'Normal',  # Nilai default
    'Sensitivity': sensitivity_input,
    'Concern': concern_input
}])

# Lakukan dummy encoding untuk input baru
input_encoded = pd.get_dummies(input_data)

# Selaraskan kolom dengan menggunakan reindex agar sama persis dengan X saat training
X_new = input_encoded.reindex(columns=X.columns, fill_value=0)

# Standardisasi input
X_new_scaled = scaler_baru.transform(X_new)

# Prediksi menggunakan model SVM Skincare
rekomendasi_linear = model_linear.predict(X_new_scaled)[0]
rekomendasi_rbf = model_rbf.predict(X_new_scaled)[0]

# Tampilkan Hasil
print('\n=== DATA YANG DIMASUKKAN ===')
print(f'Grup Umur     : {age_input}')
print(f'Sensitivitas  : {sensitivity_input}')
print(f'Masalah Kulit : {concern_input}')

print('\n=== HASIL REKOMENDASI BAHAN AKTIF (INGREDIENTS) ===')
print(f'SVM Linear : {rekomendasi_linear}')
print(f'SVM RBF    : {rekomendasi_rbf}')

=== INPUT KARAKTERISTIK KULIT ANDA ===
Pilihan Age Group: '14-18', '19-24', '25-36', '37-45', '45+'
Masukkan Age Group: 45
Pilihan Sensitivity: 'No', 'Yes'
Apakah sensitif? (Yes/No): yes
Pilihan Concern: 'Acne', 'Aging', 'Blackheads', 'Dark Spots', 'Dull Skin', 'Pores', 'Redness', 'dark circles', 'hyperpigmentation'
Masukkan Concern: hyperpigmentation
Pilihan Effects: silakan isi bebas (contoh: 'Hydrating', 'Soothing', 'Brightening')
Masukkan Effects yang dicari: Brightening

=== DATA YANG DIMASUKKAN ===
Grup Umur     : 45
Sensitivitas  : Yes
Masalah Kulit : Hyperpigmentation

=== HASIL REKOMENDASI BAHAN AKTIF (INGREDIENTS) ===
SVM Linear : Niacinamide + Vitamin C + Alpha Arbutin
SVM RBF    : Vitamin C + Niacinamide + Tranexamic Acid
